In [ ]:
import os

from pathlib import Path
os.chdir(Path.cwd().parent)
import time
import copy
import json
import random
import warnings
import numpy as np
from pathlib import Path
from collections import defaultdict
from IPython.core.magic import register_cell_magic

from data.dataloader import load_dataset
from utils.utils import set_random_seed, Usage
from utils.llm import load_model
from utils.metrics import get_metrics_fn
from utils.multiprocess import multiprocess_call
from modules.post_process import get_label_tokenizer, get_logit, prepare_labels, match_drug_fuzz
from TherapyReason.utils.significance import get_significance
from TherapyReason.therapy import get_dl_candidates, get_kg_candidates, get_therapy, rank_dl_candidates
from TherapyReason.overview import process_solution_overview_wrap
from TherapyReason.reasoning import candidate2string, template_solution_dl, solution2string_with_overview, template_solution_system, \
    post_process_solution_wrapper, template_solution_kg, candidate_similarity

set_random_seed(42)
dataset = 'mimic3'
task = 'drugrec'
label_key = 'drugs'
datasets, full_set = load_dataset(dataset, task)
test_set = datasets['test']
label_tokenizer = get_label_tokenizer(full_set, label_key)
cond_tokenizer = get_label_tokenizer(full_set, 'conditions')
proc_tokenizer = get_label_tokenizer(full_set, 'procedures')
drug_tokenizer = get_label_tokenizer(full_set, 'drugs')
significance = get_significance(dataset, task)
cache_dir = f"./TherapyReason/cache/{dataset}-{task}"
os.makedirs(cache_dir, exist_ok=True)
concept_similarity = json.loads(Path("TherapyReason/cache/llm_drug_pairwise.json").read_text())
print(len(test_set))

warnings.filterwarnings('ignore')


@register_cell_magic
def skip(line, cell): return None


llm = load_model({"model": "DeepSeek-V3-671B", "server": "ant", "do_sample": False, "temperature": 0.0, "max_attempts": 10})

In [ ]:
# Discovery
thres = 0.4
model_name = 'StageNet'
sample_dl_candidates = get_dl_candidates(thres, os.path.join(cache_dir, "sensitivity"), dataset, task, full_set, test_set, model_name)
sample_candidate_ranks = rank_dl_candidates(os.path.join(cache_dir, "sensitivity"), dataset, task, full_set, test_set, model_name)
sample_kg_candidates = get_kg_candidates(cache_dir, task, test_set, label_tokenizer, label_tokenizer)

In [ ]:
# Therapy (or Solution)
solution_file = Path(os.path.join(cache_dir, "sensitivity", f"solution_thres{int(thres * 10)}.jsonl"))
sample_solutions = get_therapy(solution_file, task, significance, label_key, test_set, sample_dl_candidates, sample_kg_candidates)

In [ ]:
# solution overview
process_solution_overview = process_solution_overview_wrap(
    os.path.join(cache_dir, "sensitivity", f"solution_thres{int(thres * 10)}_overview", llm.genre), llm)
solution_overview_output = multiprocess_call(process_solution_overview, sample_solutions,
                                             n_workers=8, verbose=False)

## solution reasoning

In [ ]:
# 初始化solution reasoning的cache文件（写空文件）
n_shot = 2
chunk_size = 20
top_p_kg = 60

for key, item in enumerate(list(zip(sample_solutions, solution_overview_output))):
    solution_obj, solution_overview = item
    visits = solution_obj['solutions']

    cache_dl_file = Path(os.path.join(cache_dir, "sensitivity", f"solution_thres{int(thres * 10)}_dl", llm.genre, f"{key}.json"))
    os.makedirs(os.path.dirname(cache_dl_file), exist_ok=True)
    if not cache_dl_file.exists():
        cached = []
        for visit_id, visit in enumerate(visits):
            _cached = []
            for solution_id, solution in enumerate(visit):
                n_chunks = int(np.ceil(len(solution["dl_candidates"]) / chunk_size))
                _cached.append({"dl_candidates": solution["dl_candidates"], "dl_query": [None] * n_chunks, "dl_response": [None] * n_chunks,
                    "dl_prediction": {}, "dl_usage": Usage().dict()})
            cached.append(_cached)
        cache_dl_file.write_text(json.dumps(cached, indent=4), encoding="utf-8")
        print(f"Init {cache_dl_file}")

    cache_kg_file = Path(os.path.join(cache_dir, "sensitivity", f"solution_thres{int(thres * 10)}_kg_{n_shot}shot", llm.genre, f"{key}.json"))
    os.makedirs(os.path.dirname(cache_kg_file), exist_ok=True)
    if not cache_kg_file.exists():
        cached = []
        for visit_id, visit in enumerate(visits):
            _cached = []
            for solution_id, solution in enumerate(visit):
                n_chunks = int(np.ceil(len(solution["kg_candidates"]) / chunk_size))
                _cached.append({"kg_candidates": solution["kg_candidates"], "kg_query": [None] * n_chunks, "kg_response": [None] * n_chunks,
                                "kg_prediction": {},"kg_usage": Usage().dict()})
            cached.append(_cached)

        cache_kg_file.write_text(json.dumps(cached, indent=4), encoding="utf-8")
        print(f"Init {cache_kg_file}")

In [ ]:
# 对DL进行reasoning
def process_solutions_dl(key, item, **kwargs):
    solution_obj, solution_overview = item
    visits = solution_obj['solutions']

    cache_file = Path(os.path.join(cache_dir, "sensitivity", f"solution_thres{int(thres * 10)}_dl", llm.genre, f"{key}.json"))
    cached = json.loads(cache_file.read_text(encoding="utf-8"))
    for visit_id, visit in enumerate(visits):
        for solution_id, solution in enumerate(visit):
            overview = solution_overview[visit_id][solution_id]['overview']

            n_chunks = int(np.ceil(len(solution['dl_candidates']) / chunk_size).item())
            for i in range(n_chunks):
                if cached[visit_id][solution_id]['dl_response'][i] is not None: continue

                time.sleep(random.randint(0, 4))
                print(f"Patient: {key}\tVisit: {visit_id + 1}/{len(visits)}\tSolution: {solution_id + 1}/{len(visit)}\tDL Chunk: {i}/{n_chunks}")
                candidate_string_dict = candidate2string(solution["dl_candidates"][i * chunk_size: (i + 1) * chunk_size], solution)

                post_process_solution = post_process_solution_wrapper(drug_tokenizer, candidates=list(candidate_string_dict))
                query = template_solution_dl.format(solution_string=solution2string_with_overview(solution, overview, overview_only=True),
                                                    candidate_dict=json.dumps(candidate_string_dict, indent=2), candidates="\t".join(candidate_string_dict))
                _, (response, prediction), _, usage = llm.understand(query, system=template_solution_system, post_process=post_process_solution,
                                                                     **kwargs)
                cached[visit_id][solution_id]['dl_query'][i] = query
                cached[visit_id][solution_id]['dl_response'][i] = response
                cached[visit_id][solution_id]['dl_prediction'].update(prediction)
                cached[visit_id][solution_id]['dl_usage'] = (usage + cached[visit_id][solution_id]['dl_usage']).dict()
                cache_file.write_text(json.dumps(cached, indent=4), encoding="utf-8")

    return key, cached


sample_dl_reasoning = multiprocess_call(process_solutions_dl, list(zip(sample_solutions, solution_overview_output)), n_workers=16, verbose=False)

In [ ]:
# 对KG进行reasoning
def process_solutions_kg(key, item, **kwargs):
    solution_obj, solution_overview, candidate_ranks = item
    visits = solution_obj['solutions']

    cache_dl_file = Path(os.path.join(cache_dir, "sensitivity", f"solution_thres{int(thres * 10)}_dl", llm.genre, f"{key}.json"))
    cache_file = Path(os.path.join(cache_dir, "sensitivity", f"solution_thres{int(thres * 10)}_kg_{n_shot}shot", llm.genre, f"{key}.json"))
    dl_cached = json.loads(cache_dl_file.read_text(encoding="utf-8"))
    cached = json.loads(cache_file.read_text(encoding="utf-8"))
    for visit_id, visit in enumerate(visits):
        for solution_id, solution in enumerate(visit):
            overview = solution_overview[visit_id][solution_id]['overview']

            # 将kg drug降序排序
            solution['kg_candidates'] = [x for x in candidate_ranks if x in solution['kg_candidates']]
            solution['kg_candidates'] = solution['kg_candidates'][:top_p_kg]

            n_chunks = int(np.ceil(len(solution['kg_candidates']) / chunk_size).item())
            for i in range(n_chunks):
                if cached[visit_id][solution_id]['kg_response'][i] is not None: continue

                time.sleep(random.randint(0, 4))
                print(f"Patient: {key}\tVisit: {visit_id + 1}/{len(visits)}\tSolution: {solution_id + 1}/{len(visit)}\tKG Chunk: {i}/{n_chunks}")
                kg_candidates = solution["kg_candidates"][i * chunk_size: (i + 1) * chunk_size]
                candidate_string_dict = candidate2string(kg_candidates, solution)
                assert candidate_string_dict

                # Use top-5 most similar DL drugs as few-shot examples
                dl_prediction = dl_cached[visit_id][solution_id]['dl_prediction']
                fewshot_similarities = [(key, candidate_similarity(key, kg_candidates, concept_similarity)) for key, value in dl_prediction.items()]
                fewshot_similarity_sorted = sorted(fewshot_similarities, key=lambda x: x[1], reverse=True)
                fewshots = {key: dl_prediction[key] for key, score in fewshot_similarity_sorted[:n_shot]}

                post_process_solution = post_process_solution_wrapper(drug_tokenizer, candidates=list(candidate_string_dict))
                query = template_solution_kg.format(solution_string=solution2string_with_overview(solution, overview, overview_only=True),
                                                    candidate_dict=json.dumps(candidate_string_dict, indent=2),
                                                    few_shot_dict=json.dumps(fewshots, indent=2))
                _, (response, prediction), _, usage = llm.understand(query, system=template_solution_system, post_process=post_process_solution,
                                                                     **kwargs)
                cached[visit_id][solution_id]['kg_query'][i] = query
                cached[visit_id][solution_id]['kg_response'][i] = response
                cached[visit_id][solution_id]['kg_prediction'].update(prediction)
                cached[visit_id][solution_id]['kg_usage'] = (usage + cached[visit_id][solution_id]['kg_usage']).dict()
                cache_file.write_text(json.dumps(cached, indent=4), encoding="utf-8")

    return key, cached


sample_kg_reasoning = multiprocess_call(process_solutions_kg, list(zip(sample_solutions, solution_overview_output, sample_candidate_ranks)), n_workers=16, verbose=False)

# eval

In [ ]:
offset = 3
threshold = 4

In [ ]:
# 评估DL的结果
def get_dl_metric(offset, threshold, dl_reasoning, agg='mean'):
    y_true_all, y_prob_all = [], []
    for i in range(len(test_set)):
        patient_prediction = defaultdict(list)
        for visit_id, visit in enumerate(dl_reasoning[i]):
            for solution_id in range(len(visit)):
                for drug, value in dl_reasoning[i][visit_id][solution_id]['dl_prediction'].items():
                    assert drug is not None, i
                    patient_prediction[drug].append(value['predict'] + offset)

        patient_prediction = {drug: np.mean(scores) if agg == 'mean' else np.max(scores) for drug, scores in patient_prediction.items()}
        patient_prediction = [drug for drug, score in patient_prediction.items() if score >= threshold]
        y_true_all.append(prepare_labels(task, [test_set[i][label_key]], label_tokenizer)[0].numpy())
        y_prob_all.append(get_logit(task, list(patient_prediction), label_tokenizer))

    metrics = get_metrics_fn(task)(np.stack(y_true_all), np.array(y_prob_all))
    print(offset, threshold, agg, metrics["precision_samples"], metrics["recall_samples"], metrics["f1_samples"], metrics["jaccard_samples"], metrics['hamming_loss'], metrics['rank'])

for agg in ['max', 'mean']:
    get_dl_metric(offset, threshold, sample_dl_reasoning, agg=agg)

In [ ]:
# 评估DL+KG的结果
def get_dlkg_metric(offset, threshold, dl_reasoning, kg_reasoning, agg='mean'):
    y_true_all, y_prob_all = [], []
    for i in range(len(test_set)):
        patient_prediction = defaultdict(list)
        for visit_id, visit in enumerate(dl_reasoning[i]):
            for solution_id in range(len(visit)):
                for drug, value in dl_reasoning[i][visit_id][solution_id]['dl_prediction'].items():
                    patient_prediction[drug].append(value['predict'] + offset)
                for drug, value in kg_reasoning[i][visit_id][solution_id]['kg_prediction'].items():
                    patient_prediction[drug].append(value['predict'])

        patient_prediction = {drug: np.mean(scores) if agg == 'mean' else np.max(scores) for drug, scores in patient_prediction.items()}
        patient_prediction = [drug for drug, score in patient_prediction.items() if score >= threshold]
        y_true_all.append(prepare_labels(task, [test_set[i][label_key]], label_tokenizer)[0].numpy())
        y_prob_all.append(get_logit(task, list(patient_prediction), label_tokenizer))

    metrics = get_metrics_fn(task)(np.stack(y_true_all), np.array(y_prob_all))
    print(offset, threshold, agg, metrics["precision_samples"], metrics["recall_samples"], metrics["f1_samples"], metrics["jaccard_samples"], metrics['hamming_loss'], metrics['rank'])

for agg in ['mean']:
    get_dlkg_metric(offset, threshold, sample_dl_reasoning, sample_kg_reasoning, agg=agg)

In [ ]:
samples = []
y_true_all, y_prob_all = [], []
for i in range(len(test_set)):
    patient_prediction = defaultdict(list)
    for visit_id, visit in enumerate(sample_dl_reasoning[i]):
        for solution_id in range(len(visit)):
            for drug, value in sample_dl_reasoning[i][visit_id][solution_id]['dl_prediction'].items():
                patient_prediction[drug].append(value['predict'] + offset)
            for drug, value in sample_kg_reasoning[i][visit_id][solution_id]['kg_prediction'].items():
                patient_prediction[drug].append(value['predict'])

    patient_prediction = {drug: np.mean(scores) for drug, scores in patient_prediction.items()}
    patient_prediction = [drug for drug, score in patient_prediction.items() if score >= threshold]

    labels = test_set[i][label_key]
    if len(labels) < 3: continue
    acc = len(set(labels).intersection(set(patient_prediction))) / len(set(labels).union(set(patient_prediction)))
    samples.append((i, acc))

samples = sorted(samples, key=lambda tuple: tuple[1], reverse=True)

In [ ]:
samples[:5]

In [ ]:
import re
selected = samples[:20]

llm = load_model({"model": "Qwen3-235B-A22B-Instruct-2507", "server": "ant3", "do_sample": False, "temperature": 0.0, "max_attempts": 5})

sample_scores = []
for i, acc in selected:
    for visit_id, visit in enumerate(sample_solutions[i]['solutions']):
        solution_names = []
        if len(visit) < 3:
            continue
        for solution_id, solution in enumerate(visit):
            if len(solution['conditions']) <= 3: continue
            solution_name = ""
            context = solution['conditions'] + solution['procedures'] + solution['drugs']
            if solution['conditions']:
                solution_name += "### Conditions\n" + "".join(x + "\n" for x in solution['conditions'])
            if solution['procedures']:
                solution_name += "### Procedures\n" + "".join(x + "\n" for x in solution['procedures'])
            solution_names.append(solution_name)

        if len(solution_names) < 3: continue

        # evaluate the coherence within solution and difference across solutions
        prompt = """\
You are good at clinical clustering. Given clusters consisting of conditions, procedures, and medications, you can assess their coherence and independence.
**Coherence**: for each cluster, please assess whether the elements within the cluster are clinically related to a single major disease. \
For example, if a cluster contains "heart attack" and "stroke", they are semantically coherence. If a cluster \
contains "heart attack" and "diabetes", they are not semantically coherence.
**Independence**: for each pair of clusters, please assess whether the elements within the two clusters are \
semantically independent. For example, if one cluster contains "heart attack" and another cluster contains \
"diabetes", they are semantically independent. If one cluster contains "heart attack" and another cluster \
contains "heart attack", they are not semantically independent.

Please evaluate the coherence and independence of the following clusters.
{clusters}

After reasoning, provide a global decision in the following format:
Coherence: [score from 1 to 5. higher score means higher coherence]
Independence: [score from 1 to 5. higher score means higher independence]
""".format(clusters="\n\n".join(solution_names))
        _, (response),_, usage = llm.understand(prompt)
        coherence_score = re.findall(r"\d", response.split("Coherence:")[-1].split("Independence:")[0])
        independence_score = re.findall(r"\d", response.split("Independence:")[-1])
        print(i, visit_id, coherence_score, independence_score)
        sample_scores.append((i, visit_id, coherence_score, independence_score))

In [ ]:
sample_scores

In [ ]:
for sample_id, visit_id, _, _ in sample_scores:
    print(sample_id, visit_id)
    solutions = sample_solutions[sample_id]['solutions'][visit_id]
    for solution_id, solution in enumerate(solutions):
        if len(solution['conditions']) <= 3: continue
        print(solution_id, solution['conditions'], solution['procedures'])
        print()
    print()

In [ ]:
sample_id = 30
visit_id = 0

In [ ]:
# baselines
sample = test_set[sample_id]
sample_id_ = f"{sample['patient_id']}_{sample['visit_id']}"
cache_file1 = Path(os.path.join("cache/mimic3-drugrec/KARE/2shot-deepseek-base-kg", f"{sample_id_}.json"))
baseline1 = json.loads(cache_file1.read_text())
print(baseline1)
cache_file2 = Path(os.path.join("cache/mimic3-drugrec/MedReason/2shot-deepseek-base-kg", f"{sample_id_}.json"))
baseline2 = json.loads(cache_file2.read_text())
print(baseline2)
cache_file3 = Path(os.path.join("cache/mimic3-drugrec/MedRAG/2shot-deepseek-base-kg", f"{sample_id_}.json"))
baseline3 = json.loads(cache_file3.read_text())
print(baseline3)

In [ ]:
conditions = ['congestive heart failure; nonhypertensive',
              'acute and unspecified renal failure',
              'heart valve disorders',
              'cardiac dysrhythmias',
              'coronary atherosclerosis and other heart disease',
              'respiratory failure; insufficiency; arrest (adult)',
              'complications of surgical procedures or medical care',
              'skin and subcutaneous tissue infections',
              'chronic ulcer of skin',
              'acute myocardial infarction',
              'diabetes mellitus without complication',
              'bacterial infection; unspecified site']
conditions = [x.split(";")[0] for x in conditions]
print("; ".join(conditions))

In [ ]:
print("; ".join(sample['procedures'][-1]))

In [ ]:
label = test_set[sample_id][label_key]
print(len(label))

summary_0 = solution_overview_output[sample_id][visit_id][0]['overview']
print(summary_0)
print()
summary_3 = solution_overview_output[sample_id][visit_id][3]['overview']
print(summary_3)

solution0 = sample_kg_reasoning[sample_id][visit_id][0]
solution0_prediction = solution0['kg_prediction']
solution0_pred_pos = [key for key, value in solution0_prediction.items() if value['predict'] >= 4]
solution0_pred_neg = [key for key, value in solution0_prediction.items() if value['predict'] <= 2]
print(len(solution0_pred_pos), len(solution0_pred_neg))

solution3 = sample_kg_reasoning[sample_id][visit_id][3]
solution3_prediction = solution3['kg_prediction']
solution3_pred_pos = [key for key, value in solution3_prediction.items() if value['predict'] >= 4]
solution3_pred_neg = [key for key, value in solution3_prediction.items() if value['predict'] <= 2]
print(len(solution3_pred_pos), len(solution3_pred_neg))

In [ ]:
print("\n".join(sample_solutions[sample_id]['solutions'][0][3]['conditions']))

In [ ]:
solution0_pred_truepos = set(solution0_pred_pos) & set(label)
solution0_pred_trueneg = set(solution0_pred_neg) - set(label)
solution3_pred_truepos = set(solution3_pred_pos) & set(label)
solution3_pred_trueneg = set(solution3_pred_neg) - set(label)
print(len(solution0_pred_truepos), len(solution0_pred_trueneg), len(solution3_pred_truepos), len(solution3_pred_trueneg))
print(solution0_pred_truepos, solution3_pred_truepos)
print(solution0_pred_truepos & solution3_pred_truepos)

In [ ]:
drug_and_rationale_0_pos = {}
for drug in solution0_pred_truepos:
    item = solution0_prediction[drug]
    query = [q for q in solution0['kg_query'] if q and drug in q][0]
    item["query"] = query
    drug_and_rationale_0_pos[drug] = item
print(drug_and_rationale_0_pos)
print()

drug_and_rationale_3_pos = {}
for drug in solution3_pred_truepos:
    item = solution3_prediction[drug]
    query = [q for q in solution3['kg_query'] if q and drug in q][0]
    item["query"] = query
    drug_and_rationale_3_pos[drug] = item
print(drug_and_rationale_3_pos)
print()

In [ ]:
drug_and_rationale_baseline1_pos = {}
drug_and_rationale_baseline2_pos = {}
drug_and_rationale_baseline3_pos = {}
for drug in solution0_pred_truepos | solution3_pred_truepos:
    item = baseline1['prediction'][drug]
    item["query"] = [q for q in baseline1['query'] if q and drug in q][0]
    drug_and_rationale_baseline1_pos[drug] = item
for drug in solution0_pred_truepos | solution3_pred_truepos:
    item = baseline2['prediction'][drug]
    item["query"] = [q for q in baseline2['query'] if q and drug in q][0]
    drug_and_rationale_baseline2_pos[drug] = item
for drug in solution0_pred_truepos | solution3_pred_truepos:
    item = baseline3['prediction'][drug]
    item["query"] = [q for q in baseline3['query'] if q and drug in q][0]
    drug_and_rationale_baseline3_pos[drug] = item
print()

In [ ]:
drugs = list(solution0_pred_truepos | solution3_pred_truepos)
print(drugs)

In [ ]:
target = drugs[2]
print(target)

In [ ]:
print(drug_and_rationale_baseline1_pos[target]['query'])
print(drug_and_rationale_baseline1_pos[target]['reason'])
print(drug_and_rationale_baseline1_pos[target]['score'])

In [ ]:
print(drug_and_rationale_baseline2_pos[target]['query'])
print(drug_and_rationale_baseline2_pos[target]['reason'])
print(drug_and_rationale_baseline2_pos[target]['score'])

In [ ]:
print(drug_and_rationale_baseline3_pos[target]['query'])
print(drug_and_rationale_baseline3_pos[target]['reason'])
print(drug_and_rationale_baseline3_pos[target]['score'])

In [ ]:
drug_dict = drug_and_rationale_3_pos
ours_query = drug_dict[target]['query']
ours_reason = drug_dict[target]['reason']
ours_score = drug_dict[target]['predict']
print(ours_query)
print(target)
print(ours_reason)
print(ours_score)